In [ ]:
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('.setup_complete'):
    !wget -q https://raw.githubusercontent.com/yandexdataschool/Practical_RL/master/setup_colab.sh -O- | bash
    !touch .setup_complete

if type(os.environ.get("DISPLAY")) is not str or len(os.environ.get("DISPLAY")) == 0:
    !bash ../xvfb start
    os.environ['DISPLAY'] = ':1'

In [ ]:
!pip install "gymnasium[toy_text,classic_control]" -q

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
%matplotlib inline

# Homework part I
### Tabular crossentropy method (Taxi-v3)

In [ ]:
env = gym.make("Taxi-v3").env
env.reset()
n_actions = env.action_space.n
n_states = env.observation_space.n
print("n_actions =", n_actions, "n_states =", n_states)

In [ ]:
policy = np.ones((n_states, n_actions)) / n_actions

def generate_session_taxi(env, policy, t_max=10**4):
    states, actions = [], []
    total_reward = 0
    s, _ = env.reset()
    for t in range(t_max):
        a = np.random.choice(n_actions, p=policy[s])
        new_s, r, terminated, truncated, _ = env.step(a)
        states.append(s)
        actions.append(a)
        total_reward += r
        s = new_s
        if terminated or truncated:
            break
    return states, actions, total_reward

def select_elites_tabular(states_batch, actions_batch, rewards_batch, percentile=50):
    threshold = np.percentile(rewards_batch, percentile)
    elite_states, elite_actions = [], []
    for states, actions, reward in zip(states_batch, actions_batch, rewards_batch):
        if reward >= threshold:
            elite_states.extend(states)
            elite_actions.extend(actions)
    return elite_states, elite_actions

def update_policy(elite_states, elite_actions, n_states, n_actions):
    new_policy = np.ones((n_states, n_actions)) / n_actions
    for s, a in zip(elite_states, elite_actions):
        new_policy[s] = 0
        new_policy[s, a] += 1
    row_sums = new_policy.sum(axis=1, keepdims=True)
    new_policy = new_policy / row_sums
    return new_policy

### 1.1 — влияние percentile и n_sessions на производительность

In [ ]:
def train_taxi(n_sessions=100, percentile=50, n_iter=20):
    policy = np.ones((n_states, n_actions)) / n_actions
    log = []
    for i in range(n_iter):
        sessions = [generate_session_taxi(env, policy) for _ in range(n_sessions)]
        states_batch, actions_batch, rewards_batch = zip(*sessions)
        states_batch = list(states_batch)
        actions_batch = list(actions_batch)
        rewards_batch = np.array(rewards_batch)
        elite_states, elite_actions = select_elites_tabular(
            states_batch, actions_batch, rewards_batch, percentile
        )
        policy = update_policy(elite_states, elite_actions, n_states, n_actions)
        log.append(np.mean(rewards_batch))
    return log

configs = [
    {"n_sessions": 100, "percentile": 50},
    {"n_sessions": 100, "percentile": 70},
    {"n_sessions": 100, "percentile": 90},
    {"n_sessions": 200, "percentile": 70},
]

plt.figure(figsize=(10, 5))
for cfg in configs:
    log = train_taxi(**cfg)
    label = f"sessions={cfg['n_sessions']}, pct={cfg['percentile']}"
    plt.plot(log, label=label)

plt.xlabel("iteration")
plt.ylabel("mean reward")
plt.title("Taxi-v3: влияние гиперпараметров")
plt.legend()
plt.grid()
plt.show()

### 1.2 — тюнинг до положительного среднего reward

In [ ]:
policy = np.ones((n_states, n_actions)) / n_actions
log = []
n_sessions = 300
percentile = 75

for i in range(100):
    sessions = [generate_session_taxi(env, policy) for _ in range(n_sessions)]
    states_batch, actions_batch, rewards_batch = zip(*sessions)
    states_batch = list(states_batch)
    actions_batch = list(actions_batch)
    rewards_batch = np.array(rewards_batch)
    elite_states, elite_actions = select_elites_tabular(
        states_batch, actions_batch, rewards_batch, percentile
    )
    policy = update_policy(elite_states, elite_actions, n_states, n_actions)
    mean_r = np.mean(rewards_batch)
    log.append(mean_r)
    if (i+1) % 10 == 0:
        print(f"iter {i+1}, mean reward = {mean_r:.2f}")
    if mean_r > 0:
        print(f"Positive mean reward reached at iteration {i+1}!")
        break

plt.plot(log)
plt.xlabel("iteration")
plt.ylabel("mean reward")
plt.title("Taxi-v3: n_sessions=300, percentile=75")
plt.grid()
plt.show()

# Homework part II
### Deep crossentropy method (CartPole-v1)

In [ ]:
env = gym.make("CartPole-v1", render_mode="rgb_array").env
env.reset()
n_actions = env.action_space.n
state_dim = env.observation_space.shape[0]

plt.imshow(env.render())
print("state vector dim =", state_dim)
print("n_actions =", n_actions)
env.close()

In [ ]:
from sklearn.neural_network import MLPClassifier

agent = MLPClassifier(
    hidden_layer_sizes=(20, 20),
    activation="tanh",
)

env = gym.make("CartPole-v1", render_mode="rgb_array").env
agent.partial_fit([env.reset()[0]] * n_actions, range(n_actions), classes=range(n_actions))

In [ ]:
def generate_session(env, agent, t_max=1000):
    states, actions = [], []
    total_reward = 0
    s, _ = env.reset()

    for t in range(t_max):
        probs = agent.predict_proba([s])[0]

        assert probs.shape == (env.action_space.n,), "make sure probabilities are a vector"

        a = np.random.choice(env.action_space.n, p=probs)

        new_s, r, terminated, truncated, _ = env.step(a)

        states.append(s)
        actions.append(a)
        total_reward += r

        s = new_s
        if terminated or truncated:
            break
    return states, actions, total_reward

In [ ]:
dummy_states, dummy_actions, dummy_reward = generate_session(env, agent, t_max=5)
print("states:", np.stack(dummy_states))
print("actions:", dummy_actions)
print("reward:", dummy_reward)

In [ ]:
def select_elites(states_batch, actions_batch, rewards_batch, percentile=50):
    threshold = np.percentile(rewards_batch, percentile)
    elite_states, elite_actions = [], []
    for states, actions, reward in zip(states_batch, actions_batch, rewards_batch):
        if reward >= threshold:
            elite_states.extend(states)
            elite_actions.extend(actions)
    return elite_states, elite_actions

In [ ]:
def show_progress(rewards_batch, log, percentile, reward_range=[-990, +10]):
    mean_reward = np.mean(rewards_batch)
    threshold = np.percentile(rewards_batch, percentile)
    log.append([mean_reward, threshold])

    clear_output(True)
    print("mean reward = %.3f, threshold=%.3f" % (mean_reward, threshold))
    plt.figure(figsize=[8, 4])
    plt.subplot(1, 2, 1)
    plt.plot(list(zip(*log))[0], label="Mean rewards")
    plt.plot(list(zip(*log))[1], label="Reward thresholds")
    plt.legend()
    plt.grid()

    plt.subplot(1, 2, 2)
    plt.hist(rewards_batch, range=reward_range)
    plt.vlines(
        [np.percentile(rewards_batch, percentile)],
        [0],
        [100],
        label="percentile",
        color="red",
    )
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
n_sessions = 100
percentile = 70
log = []

for i in range(100):
    sessions = [generate_session(env, agent) for _ in range(n_sessions)]

    states_batch, actions_batch, rewards_batch = zip(*sessions)
    states_batch = list(states_batch)
    actions_batch = list(actions_batch)
    rewards_batch = np.array(rewards_batch)

    elite_states, elite_actions = select_elites(states_batch, actions_batch, rewards_batch, percentile)

    agent.partial_fit(elite_states, elite_actions)

    show_progress(
        rewards_batch, log, percentile, reward_range=[0, np.max(rewards_batch)]
    )

    if np.mean(rewards_batch) > 190:
        print("You Win! You may stop training now via KeyboardInterrupt.")

# Results

In [ ]:
from gymnasium.wrappers import RecordVideo

with RecordVideo(
    env=gym.make("CartPole-v1", render_mode="rgb_array"),
    video_folder="./videos",
    episode_trigger=lambda episode_number: True,
) as env_monitor:
    sessions = [generate_session(env_monitor, agent) for _ in range(100)]

In [ ]:
from pathlib import Path
from base64 import b64encode
from IPython.display import HTML

video_paths = sorted([s for s in Path("videos").iterdir() if s.suffix == ".mp4"])
video_path = video_paths[-1]

if "google.colab" in sys.modules:
    with video_path.open("rb") as fp:
        mp4 = fp.read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
else:
    data_url = str(video_path)

HTML("""
<video width="640" height="480" controls>
  <source src="{}" type="video/mp4">
</video>
""".format(data_url))

### 2.1 — MountainCar-v0 (цель: average reward >= -150)

In [ ]:
env_mc = gym.make("MountainCar-v0").env
n_actions_mc = env_mc.action_space.n
state_dim_mc = env_mc.observation_space.shape[0]
print("state_dim =", state_dim_mc, "n_actions =", n_actions_mc)

agent_mc = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    activation="tanh",
    learning_rate_init=0.001,
    max_iter=1,
    warm_start=True,
)

agent_mc.partial_fit(
    [env_mc.reset()[0]] * n_actions_mc,
    range(n_actions_mc),
    classes=range(n_actions_mc)
)

In [ ]:
def generate_session_mc(env, agent, t_max=10000):
    states, actions = [], []
    total_reward = 0
    s, _ = env.reset()

    for t in range(t_max):
        probs = agent.predict_proba([s])[0]
        a = np.random.choice(env.action_space.n, p=probs)
        new_s, r, terminated, truncated, _ = env.step(a)

        states.append(s)
        actions.append(a)
        total_reward += r

        s = new_s
        if terminated or truncated:
            break
    return states, actions, total_reward

### 2.2 — повторное использование сэмплов из последних N итераций

In [ ]:
n_sessions_mc = 100
percentile_mc = 70
log_mc = []
n_last = 4

all_states_history = []
all_actions_history = []
all_rewards_history = []

for i in range(100):
    sessions = [generate_session_mc(env_mc, agent_mc) for _ in range(n_sessions_mc)]

    states_b, actions_b, rewards_b = zip(*sessions)
    states_b = list(states_b)
    actions_b = list(actions_b)
    rewards_b = list(rewards_b)

    all_states_history.append(states_b)
    all_actions_history.append(actions_b)
    all_rewards_history.append(rewards_b)

    combined_states = sum(all_states_history[-n_last:], [])
    combined_actions = sum(all_actions_history[-n_last:], [])
    combined_rewards = sum(all_rewards_history[-n_last:], [])

    elite_states, elite_actions = select_elites(
        combined_states, combined_actions, combined_rewards, percentile_mc
    )

    agent_mc.partial_fit(elite_states, elite_actions)

    mean_r = np.mean(rewards_b)
    threshold = np.percentile(rewards_b, percentile_mc)
    log_mc.append([mean_r, threshold])

    clear_output(True)
    print(f"iter {i+1}: mean={mean_r:.1f}, threshold={threshold:.1f}")

    plt.figure(figsize=[8, 4])
    plt.subplot(1, 2, 1)
    plt.plot([x[0] for x in log_mc], label="Mean")
    plt.plot([x[1] for x in log_mc], label="Threshold")
    plt.axhline(-150, color='green', linestyle='--', label="Target")
    plt.legend()
    plt.grid()
    plt.subplot(1, 2, 2)
    plt.hist(rewards_b, range=[-10000, 0])
    plt.vlines([threshold], [0], [50], color="red", label="threshold")
    plt.legend()
    plt.grid()
    plt.show()

    if mean_r >= -150:
        print(f"Target reached! mean reward = {mean_r:.1f}")
        break

### 2.2 — параллельная генерация сессий через joblib

In [ ]:
from joblib import Parallel, delayed

def generate_session_parallel(agent, t_max=10000):
    env = gym.make("MountainCar-v0").env
    states, actions = [], []
    total_reward = 0
    s, _ = env.reset()

    for t in range(t_max):
        probs = agent.predict_proba([s])[0]
        a = np.random.choice(env.action_space.n, p=probs)
        new_s, r, terminated, truncated, _ = env.step(a)
        states.append(s)
        actions.append(a)
        total_reward += r
        s = new_s
        if terminated or truncated:
            break
    env.close()
    return states, actions, total_reward

n_jobs = 4
sessions_parallel = Parallel(n_jobs=n_jobs)(
    delayed(generate_session_parallel)(agent_mc) for _ in range(20)
)
print("Parallel sessions mean reward:", np.mean([s[2] for s in sessions_parallel]))

In [ ]:
def visualize_mountain_car(env, agent):
    xs = np.linspace(env.min_position, env.max_position, 100)
    vs = np.linspace(-env.max_speed, env.max_speed, 100)

    grid = np.dstack(np.meshgrid(xs, vs[::-1])).transpose(1, 0, 2)
    grid_flat = grid.reshape(len(xs) * len(vs), 2)
    probs = (
        agent.predict_proba(grid_flat).reshape(len(xs), len(vs), 3).transpose(1, 0, 2)
    )

    f, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(
        probs,
        extent=(env.min_position, env.max_position, -env.max_speed, env.max_speed),
        aspect="auto",
    )
    ax.set_title("Learned policy: red=left, green=nothing, blue=right")
    ax.set_xlabel("position (x)")
    ax.set_ylabel("velocity (v)")

    states, actions, _ = generate_session_mc(env, agent)
    states = np.array(states)
    ax.plot(states[:, 0], states[:, 1], color="white")

    for (x, v), a in zip(states[::3], actions[::3]):
        if a == 0:
            plt.arrow(x, v, -0.1, 0, color="white", head_length=0.02)
        elif a == 2:
            plt.arrow(x, v, 0.1, 0, color="white", head_length=0.02)

with gym.make("MountainCar-v0", render_mode="rgb_array").env as env:
    visualize_mountain_car(env, agent_mc)